In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

print("="*60)
print("    🏥 MEDICAL COST PREDICTION PROJECT")
print("="*60)
print("✅ All libraries imported successfully!")

In [ ]:
# Load the dataset
data = pd.read_csv('insurance.csv')

print("\n📊 DATASET LOADED SUCCESSFULLY!")
print("="*60)
print(f"Total Records: {len(data)}")
print(f"Total Features: {len(data.columns)}")
print("\n📋 First 5 Records:")
print(data.head())

In [ ]:
# Display dataset information
print("\n📊 DATASET INFORMATION:")
print("="*60)
print(data.info())

print("\n📈 STATISTICAL SUMMARY:")
print("="*60)
print(data.describe())

print("\n🔍 MISSING VALUES CHECK:")
print("="*60)
print(data.isnull().sum())
print("\n✅ No missing values found!" if data.isnull().sum().sum() == 0 else "⚠️ Missing values detected!")

In [ ]:
# Create a copy for processing
data_processed = data.copy()

print("\n🔧 DATA PREPROCESSING STARTED...")
print("="*60)

# Convert categorical variables to numerical
# Sex: male=1, female=0
data_processed['sex'] = data_processed['sex'].map({'male': 1, 'female': 0})
print("✅ Sex converted: male=1, female=0")

# Smoker: yes=1, no=0
data_processed['smoker'] = data_processed['smoker'].map({'yes': 1, 'no': 0})
print("✅ Smoker converted: yes=1, no=0")

# Region: Convert to dummy variables (one-hot encoding)
region_dummies = pd.get_dummies(data_processed['region'], prefix='region', drop_first=True)
data_processed = pd.concat([data_processed, region_dummies], axis=1)
data_processed.drop('region', axis=1, inplace=True)
print("✅ Region converted to numerical values")

print("\n📊 PROCESSED DATASET:")
print(data_processed.head())
print("\n✅ DATA PREPROCESSING COMPLETED!")

In [ ]:
# Separate features (X) and target (y)
X = data_processed.drop('charges', axis=1)  # Independent variables
y = data_processed['charges']  # Dependent variable (what we want to predict)

print("\n🎯 FEATURE ENGINEERING:")
print("="*60)
print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

print("\n📋 Feature Names:")
print(list(X.columns))

print("\n✅ Features and target separated successfully!")

In [ ]:
# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\n✂️ TRAIN-TEST SPLIT:")
print("="*60)
print(f"Training set size: {X_train.shape[0]} records ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set size: {X_test.shape[0]} records ({X_test.shape[0]/len(X)*100:.1f}%)")
print("\n✅ Data split completed!")

In [ ]:
# Create and train the model
print("\n🎓 MODEL TRAINING STARTED...")
print("="*60)

model = LinearRegression()
model.fit(X_train, y_train)

print("✅ Model trained successfully!")

# Display model parameters
print("\n📊 MODEL PARAMETERS:")
print(f"Intercept (bias): ${model.intercept_:,.2f}")

print("\n📋 Feature Coefficients:")
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)

print(coefficients.to_string(index=False))

In [ ]:
# Make predictions on test data
print("\n🔮 MAKING PREDICTIONS...")
print("="*60)

y_pred = model.predict(X_test)

# Convert to numpy array to avoid future issues
y_pred = np.array(y_pred)
y_test_values = y_test.values if hasattr(y_test, 'values') else y_test

# Display first 10 predictions
comparison = pd.DataFrame({
    'Actual Cost': y_test_values[:10],
    'Predicted Cost': y_pred[:10],
    'Difference': y_test_values[:10] - y_pred[:10]
})

print("\n📊 FIRST 10 PREDICTIONS:")
print(comparison.to_string(index=False))
print("\n✅ Predictions completed!")

In [ ]:
# Calculate performance metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Calculate accuracy within 20% tolerance (more meaningful for regression)
try:
    y_test_array = y_test.values if hasattr(y_test, 'values') else y_test
    accuracy_within_20pct = np.mean(np.abs((y_test_array - y_pred) / y_test_array) <= 0.20) * 100
except:
    accuracy_within_20pct = "N/A"

print("\n📈 MODEL EVALUATION RESULTS:")
print("="*60)
print(f"✅ R² Score: {r2:.4f}")
if isinstance(accuracy_within_20pct, (int, float)):
    print(f"✅ Predictions within 20% of actual: {accuracy_within_20pct:.1f}%")
print(f"✅ Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"✅ Mean Absolute Error (MAE): ${mae:,.2f}")
print("="*60)

# Interpretation
print("\n📝 INTERPRETATION:")
if r2 >= 0.75:
    print("✅ Model performance is GOOD!")
elif r2 >= 0.60:
    print("⚠️ Model performance is MODERATE")
else:
    print("❌ Model performance needs improvement")

print(f"\n💡 The model can explain {r2*100:.2f}% of the variance in medical costs.")
print(f"💡 On average, predictions are off by ${mae:,.2f}")

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

print("\n📊 CREATING INTERACTIVE VISUALIZATION...")
print("="*60)

# Check if we need to recreate any missing variables
if 'y_test' not in locals() and 'y_test' not in globals():
    print("⚠️ y_test not found. You need to run the previous cells first.")
    print("Please run cells 1-9 before running this visualization cell.")
else:
    # Ensure y_test_values is properly defined
    if 'y_test_values' not in locals() and 'y_test_values' not in globals():
        y_test_values = y_test.values if hasattr(y_test, 'values') else y_test
    
    # Check for other required variables
    required_vars = ['y_pred', 'r2', 'rmse', 'mae']
    missing_vars = [var for var in required_vars if var not in locals() and var not in globals()]
    
    if missing_vars:
        print(f"⚠️ Missing variables: {missing_vars}")
        print("Please run the previous cells (1-9) to train the model and make predictions.")
    else:
        # Create a copy of the comparison DataFrame for visualization
        plot_data = pd.DataFrame({
            'Actual Cost': y_test_values,
            'Predicted Cost': y_pred
        })
        
        # Sort by Actual Cost for better visualization
        plot_data_sorted = plot_data.sort_values('Actual Cost').reset_index(drop=True)
        
        # Create index for x-axis
        patient_index = np.arange(len(plot_data_sorted))
        
        # Create figure
        fig = go.Figure()
        
        # Add Actual Cost trace
        fig.add_trace(go.Scatter(
            x=patient_index,
            y=plot_data_sorted['Actual Cost'],
            mode='markers+lines',
            name='Actual Cost',
            line=dict(color='#2196F3', width=2),
            marker=dict(color='#2196F3', size=8, symbol='circle'),
            opacity=0.7,
            hovertemplate='Patient: %{x}<br>Actual: $%{y:,.0f}<extra></extra>'
        ))
        
        # Add Predicted Cost trace
        fig.add_trace(go.Scatter(
            x=patient_index,
            y=plot_data_sorted['Predicted Cost'],
            mode='markers+lines',
            name='Predicted Cost',
            line=dict(color='#F44336', width=2),
            marker=dict(color='#F44336', size=8, symbol='square'),
            opacity=0.7,
            hovertemplate='Patient: %{x}<br>Predicted: $%{y:,.0f}<extra></extra>'
        ))
        
        # Add Prediction Error area (optional)
        fig.add_trace(go.Scatter(
            x=list(patient_index) + list(patient_index[::-1]),
            y=list(plot_data_sorted['Actual Cost']) + list(plot_data_sorted['Predicted Cost'][::-1]),
            fill='toself',
            fillcolor='rgba(128, 0, 128, 0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            name='Prediction Error',
            hoverinfo='skip'
        ))
        
        # Update layout
        fig.update_layout(
            title=dict(
                text=f"Medical Cost Prediction Results<br>R² Score: {r2:.4f} | Variance Explained: {r2*100:.2f}%",
                font=dict(size=16, weight='bold'),
                x=0.5
            ),
            xaxis=dict(
                title="Patient Index (sorted by actual cost)",
                gridcolor='lightgray',
                gridwidth=1
            ),
            yaxis=dict(
                title="Medical Cost ($)",
                gridcolor='lightgray',
                gridwidth=1
            ),
            plot_bgcolor='white',
            hovermode='x unified',
            showlegend=True,
            legend=dict(
                bgcolor='rgba(255, 255, 255, 0.8)',
                bordercolor='black',
                borderwidth=1,
                x=0.02,
                y=0.98
            ),
            annotations=[
                dict(
                    x=0.98,
                    y=0.02,
                    xref='paper',
                    yref='paper',
                    text=f"RMSE: ${rmse:,.2f}<br>MAE: ${mae:,.2f}<br>Samples: {len(y_test)}",
                    showarrow=False,
                    align='right',
                    bgcolor='wheat',
                    bordercolor='black',
                    borderwidth=1,
                    borderpad=4,
                    font=dict(size=10)
                )
            ]
        )
        
        # Display the figure
        fig.show()
        
        print("\n📊 INTERACTIVE VISUALIZATION CREATED!")
        print("="*60)
        print("✅ Actual vs Predicted costs displayed")
        print("✅ Points sorted by actual cost for better comparison")
        print("✅ Hover over points to see exact values")

In [ ]:
def predict_medical_cost(age, sex, bmi, children, smoker, region):
    """
    Predict medical cost for a new patient
    
    Parameters:
    -----------
    age : int (18-100)
    sex : str ('male' or 'female')
    bmi : float (15-50)
    children : int (0-5)
    smoker : str ('yes' or 'no')
    region : str ('northeast', 'northwest', 'southeast', 'southwest')
    
    Returns:
    --------
    predicted_cost : float
    """
    
    # Prepare input data
    input_data = pd.DataFrame({
        'age': [age],
        'sex': [1 if sex.lower() == 'male' else 0],
        'bmi': [bmi],
        'children': [children],
        'smoker': [1 if smoker.lower() == 'yes' else 0],
        'region_northwest': [1 if region.lower() == 'northwest' else 0],
        'region_southeast': [1 if region.lower() == 'southeast' else 0],
        'region_southwest': [1 if region.lower() == 'southwest' else 0]
    })
    
    # Ensure columns match training data
    for col in X.columns:
        if col not in input_data.columns:
            input_data[col] = 0
    
    input_data = input_data[X.columns]
    
    # Make prediction
    prediction = model.predict(input_data)[0]
    
    return prediction

# Test the function
print("\n🧪 TESTING THE MODEL WITH SAMPLE PATIENTS:")
print("="*60)

# Test Case 1: Young healthy non-smoker
try:
    cost1 = predict_medical_cost(25, 'male', 22, 0, 'no', 'southwest')
    print(f"\n👤 Patient 1: 25-year-old male, BMI 22, Non-smoker")
    print(f"   💰 Predicted Cost: ${cost1:,.2f}")
except Exception as e:
    print(f"\n❌ Error with Patient 1: {e}")

# Test Case 2: Middle-aged smoker
try:
    cost2 = predict_medical_cost(45, 'female', 30, 2, 'yes', 'northeast')
    print(f"\n👤 Patient 2: 45-year-old female, BMI 30, Smoker, 2 children")
    print(f"   💰 Predicted Cost: ${cost2:,.2f}")
except Exception as e:
    print(f"\n❌ Error with Patient 2: {e}")

# Test Case 3: Older person
try:
    cost3 = predict_medical_cost(60, 'male', 28, 0, 'no', 'southeast')
    print(f"\n👤 Patient 3: 60-year-old male, BMI 28, Non-smoker")
    print(f"   💰 Predicted Cost: ${cost3:,.2f}")
except Exception as e:
    print(f"\n❌ Error with Patient 3: {e}")

print("\n" + "="*60)
print("✅ MODEL TESTING COMPLETED!")

In [ ]:
# NOW YOU TRY!
print("\n🎯 YOUR TURN TO TEST THE MODEL!")
print("="*60)

# Customize these values:
my_age = 30
my_sex = 'male'
my_bmi = 25.0
my_children = 1
my_smoker = 'no'
my_region = 'southwest'

try:
    # Get prediction
    my_cost = predict_medical_cost(my_age, my_sex, my_bmi, my_children, my_smoker, my_region)
    
    print(f"\n👤 YOUR PATIENT DETAILS:")
    print(f"   Age: {my_age} years")
    print(f"   Sex: {my_sex}")
    print(f"   BMI: {my_bmi}")
    print(f"   Children: {my_children}")
    print(f"   Smoker: {my_smoker}")
    print(f"   Region: {my_region}")
    print(f"\n💰 PREDICTED ANNUAL MEDICAL COST: ${my_cost:,.2f}")
    
    # Provide insights
    if my_smoker.lower() == 'yes':
        print("\n⚠️ WARNING: Smoking significantly increases medical costs!")
        print("   Consider quitting to reduce healthcare expenses.")
    elif my_bmi > 30:
        print("\n⚠️ NOTE: High BMI may increase medical costs.")
        print("   Maintaining a healthy weight can reduce expenses.")
    elif my_bmi < 18.5:
        print("\n⚠️ NOTE: Low BMI may indicate health issues.")
        print("   Consult with a healthcare provider.")
    else:
        print("\n✅ Great! Healthy profile for medical costs.")
        
except Exception as e:
    print(f"\n❌ ERROR making prediction: {e}")
    print("\n💡 TROUBLESHOOTING TIPS:")
    print("   1. Check that region is one of: northeast, northwest, southeast, southwest")
    print("   2. Ensure BMI is between 15-50")
    print("   3. Make sure sex is 'male' or 'female'")
    print("   4. Ensure smoker is 'yes' or 'no'")

print("\n" + "="*60)
print("🏁 PROJECT COMPLETED SUCCESSFULLY! 🎉")
print("="*60)